# Расхождения commission_monthly по agr_id (Q1 2026)

Отдельная тетрадка для поиска `agr_id`, где `commission_monthly` из Озера (R2 `tariff_fix.c_summa`) отличается от Excel.

Итоговые поля:
- `report_month`
- `agr_id`
- `commission_monthly_lake`
- `commission_monthly_excel`
- `tariff_name`

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

# Показывать все колонки и не обрезать ширину таблиц в выводе
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

In [ ]:
excel_sources = [
    {'report_month': '2026-01-01', 'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

target_agr_ids = [
    '144839809949',
    '1000150780552',  # за март не совпадает с Озером
    '1000871167243',  # данные совпадают
]

target_agr_id = target_agr_ids[0]

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 1) Загрузка Excel Q1 и нормализация commission_monthly

In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    return None


def normalize_agr_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return None
    return s


frames = []
for src in excel_sources:
    raw = pd.read_excel(src['path'], header=src['header'])

    agr_col = pick_column(raw.columns, ['ID договора', 'agr_id'])
    cm_col = pick_column(raw.columns, ['Комиссия \n(₽ в месяц)', 'Комиссия (₽ в месяц)'])

    if agr_col is None:
        raise ValueError(f"В файле {src['path']} не найдена колонка agr_id (ID договора)")
    if cm_col is None:
        raise ValueError(f"В файле {src['path']} не найдена колонка комиссии: Комиссия \\n(₽ в месяц)")

    df = pd.DataFrame({
        'agr_id': raw[agr_col].apply(normalize_agr_id),
        'commission_monthly_excel': pd.to_numeric(raw[cm_col], errors='coerce'),
    })
    df['report_month'] = pd.to_datetime(src['report_month'])
    frames.append(df)

excel_df = pd.concat(frames, ignore_index=True)
excel_df = excel_df[excel_df['agr_id'].notna()].copy()

print(f'Excel Q1 rows (with agr_id): {len(excel_df):,}')
print('Rows by month:')
print(excel_df.groupby(excel_df['report_month'].dt.strftime('%Y-%m')).size())
excel_df.head(5)

## 2) Выгрузка commission_monthly и tariff_name из Озера (R2)

In [ ]:
agr_values = sorted(excel_df['agr_id'].astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in agr_values]) if agr_values else "''"

sql_lake = f"""
select
  cast(m.id as string) as agr_id,
  cast(max(tp.c_name) as string) as tariff_name,
  max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_lake
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_tune tt
  on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_fix tf
  on cast(tf.id as string) = cast(tt.c_tariff as string)
where cast(m.id as string) in ({agr_in})
group by cast(m.id as string)
"""

sql_agreement_financial = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(max(a.cf_agr_financial) as string) as cf_agr_financial
from ods_alpha.scd1_agreements a
where cast(a.abs_agr_id as string) in ({agr_in})
group by cast(a.abs_agr_id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    lake_df = imp.fetch(sql_lake)
    financial_df = imp.fetch(sql_agreement_financial)

if lake_df is None:
    lake_df = pd.DataFrame(columns=['agr_id', 'tariff_name', 'commission_monthly_lake'])
if financial_df is None:
    financial_df = pd.DataFrame(columns=['agr_id', 'cf_agr_financial'])

if not lake_df.empty:
    lake_df['agr_id'] = lake_df['agr_id'].astype(str)
    lake_df['tariff_name'] = lake_df['tariff_name'].astype('string')
    lake_df['commission_monthly_lake'] = pd.to_numeric(lake_df['commission_monthly_lake'], errors='coerce')

if not financial_df.empty:
    financial_df['agr_id'] = financial_df['agr_id'].astype(str)
    financial_df['cf_agr_financial'] = financial_df['cf_agr_financial'].astype('string')

print(f'Lake rows: {len(lake_df):,}')
print(f'Agreements financial rows: {len(financial_df):,}')
lake_df.head(5)

## 3) Строгое сравнение и отбор расхождений

In [ ]:
rows_excel = len(excel_df)

merged_df = excel_df.merge(lake_df, on='agr_id', how='left')
merged_df = merged_df.merge(financial_df, on='agr_id', how='left')
rows_after_merge = len(merged_df)

merged_df['report_month_str'] = pd.to_datetime(merged_df['report_month'], errors='coerce').dt.strftime('%Y-%m')
merged_df['cf_agr_financial'] = merged_df['cf_agr_financial'].astype('string').str.strip().str.upper()

excel_val = merged_df['commission_monthly_excel']
lake_val = merged_df['commission_monthly_lake']

# Спец-правило: NaN в lake и 0.0 в Excel считаем совпадением
nan_lake_zero_excel_match = lake_val.isna() & excel_val.notna() & (excel_val == 0)

is_diff_strict = (
    (excel_val.isna() & lake_val.notna())
    | (excel_val.notna() & lake_val.isna() & (~nan_lake_zero_excel_match))
    | (excel_val.notna() & lake_val.notna() & (excel_val != lake_val))
)

merged_df['is_diff_strict'] = is_diff_strict

result_columns = [
    'report_month',
    'agr_id',
    'commission_monthly_lake',
    'commission_monthly_excel',
    'tariff_name',
    'cf_agr_financial',
]

mismatch_df = merged_df.loc[merged_df['is_diff_strict'], result_columns].copy()
mismatch_df['report_month'] = pd.to_datetime(mismatch_df['report_month'], errors='coerce').dt.strftime('%Y-%m')
mismatch_df = mismatch_df.sort_values(['report_month', 'agr_id']).reset_index(drop=True)

# Только март: расхождения и 10 примеров с разным c_summa
mismatch_march_df = mismatch_df.loc[mismatch_df['report_month'] == '2026-03'].copy()
mismatch_march_sample_10 = mismatch_march_df.head(10).copy()

print('QC:')
print(f'  rows in excel perimeter: {rows_excel:,}')
print(f'  rows after merge:        {rows_after_merge:,}')
print(f'  strict mismatches (Q1):  {len(mismatch_df):,}')
print(f'  strict mismatches (Mar): {len(mismatch_march_df):,}')

missing_lake_cnt = merged_df.loc[merged_df['commission_monthly_lake'].isna(), 'agr_id'].nunique()
print(f'  unique agr_id without lake commission: {missing_lake_cnt:,}')

print(f'  NaN(lake) vs 0.0(Excel) treated as matches: {int(nan_lake_zero_excel_match.sum()):,}')

print('\nMismatches by month (Q1):')
print(mismatch_df.groupby('report_month').size().rename('mismatch_rows'))

# Проверка гипотезы: при расхождении значение cf_agr_financial отлично от A
mismatch_with_financial = mismatch_df.copy()
mismatch_with_financial['is_not_A'] = mismatch_with_financial['cf_agr_financial'].fillna('NULL') != 'A'

hyp_total = len(mismatch_with_financial)
hyp_not_a = int(mismatch_with_financial['is_not_A'].sum())
hyp_a = int((mismatch_with_financial['cf_agr_financial'] == 'A').sum())
hyp_null = int(mismatch_with_financial['cf_agr_financial'].isna().sum())

hyp_not_a_pct = round((hyp_not_a / hyp_total) * 100, 2) if hyp_total else 0.0
hyp_a_pct = round((hyp_a / hyp_total) * 100, 2) if hyp_total else 0.0

print('\nГипотеза cf_agr_financial != A для расхождений:')
print(f'  total mismatches: {hyp_total:,}')
print(f'  cf_agr_financial != A (including NULL): {hyp_not_a:,} ({hyp_not_a_pct}%)')
print(f'  cf_agr_financial == A: {hyp_a:,} ({hyp_a_pct}%)')
print(f'  cf_agr_financial is NULL: {hyp_null:,}')

print('\nРаспределение cf_agr_financial в mismatch:')
print(mismatch_with_financial['cf_agr_financial'].fillna('NULL').value_counts(dropna=False))

print('\nПримеры mismatch, где cf_agr_financial == A (если есть):')
print(mismatch_with_financial.loc[mismatch_with_financial['cf_agr_financial'] == 'A'].head(10))

# Проверка на максимальное значение c_summa (commission_monthly_lake) из Озера
if merged_df['commission_monthly_lake'].notna().any():
    max_idx = merged_df['commission_monthly_lake'].idxmax()
    max_row = merged_df.loc[max_idx, ['report_month_str', 'agr_id', 'commission_monthly_lake', 'commission_monthly_excel', 'tariff_name', 'cf_agr_financial']]
    max_row = max_row.rename({'report_month_str': 'report_month'}).to_frame().T

    print('\nMax c_summa from lake:')
    print(max_row)
else:
    print('\nMax c_summa from lake: not found (all values are NULL)')

# 5 примеров agr_id, где значения совпадают (включая спец-правило NaN(lake) и 0.0(Excel))
is_match = (~merged_df['is_diff_strict']) & (
    (merged_df['commission_monthly_lake'].notna() & merged_df['commission_monthly_excel'].notna())
    | nan_lake_zero_excel_match
)

matches_df = merged_df.loc[is_match, [
    'report_month', 'agr_id', 'commission_monthly_lake', 'commission_monthly_excel', 'tariff_name', 'cf_agr_financial'
]].copy()

matches_df['report_month'] = pd.to_datetime(matches_df['report_month'], errors='coerce').dt.strftime('%Y-%m')
matches_df = matches_df.sort_values(['report_month', 'agr_id']).reset_index(drop=True)

print('\nПримеры 5 agr_id с совпадением Озеро=Excel:')
print(matches_df.head(5))

print('\nМарт: 10 примеров расхождений с разным c_summa:')
print(mismatch_march_sample_10)

# Контрольная проверка по заданным agr_id
check_agr_ids = ['1000150780552', '1000871167243']
check_df = merged_df.loc[
    merged_df['agr_id'].isin(check_agr_ids),
    [
        'report_month_str', 'agr_id', 'commission_monthly_lake', 'commission_monthly_excel',
        'is_diff_strict', 'tariff_name', 'cf_agr_financial'
    ]
].copy()

check_df = check_df.rename(columns={'report_month_str': 'report_month'})
check_df = check_df.loc[check_df['report_month'] == '2026-03']
check_df = check_df.sort_values(['agr_id', 'report_month']).reset_index(drop=True)

print('\nПроверка по agr_id 1000150780552 и 1000871167243 (март):')
print(check_df)

mismatch_march_sample_10

## 4) Поиск agr_id по паттернам за 3 месяца

Блок ниже формирует 3 списка `agr_id`:
1. где Excel и Озеро совпадают все 3 месяца;
2. где в Озере есть `c_summa` все 3 месяца, а в Excel все 3 месяца `0.0`;
3. где Excel и Озеро не совпадают все 3 месяца.

In [ ]:
# Нормализованная таблица для анализа по 3 месяцам
per_month_df = merged_df[[
    'agr_id', 'report_month_str', 'commission_monthly_lake', 'commission_monthly_excel', 'is_diff_strict', 'tariff_name', 'cf_agr_financial'
]].copy()

per_month_df = per_month_df.rename(columns={'report_month_str': 'report_month'})
per_month_df = per_month_df[per_month_df['report_month'].isin(['2026-01', '2026-02', '2026-03'])].copy()

# Оставляем по одной записи на agr_id+month (если дубли в Excel, берем первую)
per_month_df = per_month_df.sort_values(['agr_id', 'report_month']).drop_duplicates(['agr_id', 'report_month'], keep='first')

# Пер-месячные флаги
per_month_df['excel_is_zero'] = per_month_df['commission_monthly_excel'].fillna(np.nan) == 0
per_month_df['lake_has_value'] = per_month_df['commission_monthly_lake'].notna()
per_month_df['is_match_month'] = ~per_month_df['is_diff_strict']
per_month_df['is_mismatch_month'] = per_month_df['is_diff_strict']

agg_3m = (
    per_month_df.groupby('agr_id', as_index=False)
    .agg(
        months_cnt=('report_month', 'nunique'),
        match_months=('is_match_month', 'sum'),
        mismatch_months=('is_mismatch_month', 'sum'),
        excel_zero_months=('excel_is_zero', 'sum'),
        lake_value_months=('lake_has_value', 'sum'),
    )
)

# Берем только полноценные кейсы с 3 месяцами
agg_3m_full = agg_3m[agg_3m['months_cnt'] == 3].copy()

agr_all_match_3m = agg_3m_full.loc[agg_3m_full['match_months'] == 3, 'agr_id'].sort_values().tolist()
agr_lake_has_excel_zero_3m = agg_3m_full.loc[
    (agg_3m_full['lake_value_months'] == 3) & (agg_3m_full['excel_zero_months'] == 3),
    'agr_id'
].sort_values().tolist()
agr_all_mismatch_3m = agg_3m_full.loc[agg_3m_full['mismatch_months'] == 3, 'agr_id'].sort_values().tolist()

print('Проверка за 3 месяца (только agr_id с полным покрытием Jan/Feb/Mar):')
print(f"  1) Совпадают Excel и Озеро все 3 месяца: {len(agr_all_match_3m):,}")
print(f"  2) В Озере есть c_summa 3/3, а в Excel 0.0 3/3: {len(agr_lake_has_excel_zero_3m):,}")
print(f"  3) Не совпадают месячные комиссии все 3 месяца: {len(agr_all_mismatch_3m):,}")

print('\nСписки agr_id:')
print('1) agr_all_match_3m =', agr_all_match_3m)
print('2) agr_lake_has_excel_zero_3m =', agr_lake_has_excel_zero_3m)
print('3) agr_all_mismatch_3m =', agr_all_mismatch_3m)

# Детализация по каждому списку для проверки глазами
def show_agr_details(title, agr_list, max_rows=60):
    print(f'\n{title}')
    if not agr_list:
        print('No agr_id')
        return
    details = per_month_df[per_month_df['agr_id'].isin(agr_list)].copy()
    details = details.sort_values(['agr_id', 'report_month'])
    print(f'Rows: {len(details):,}, unique agr_id: {details["agr_id"].nunique():,}')
    print(details.head(max_rows))

show_agr_details('Детализация: совпадают 3/3', agr_all_match_3m)
show_agr_details('Детализация: lake есть 3/3 и excel=0.0 3/3', agr_lake_has_excel_zero_3m)
show_agr_details('Детализация: mismatch 3/3', agr_all_mismatch_3m)

## 5) Диагностика по agr_id в agr_terms и тарифных таблицах

Ниже блок, который помогает глазами проверить, где может теряться/искажаться месячная сумма.
Для `target_agr_id` выводятся строки из:
- `ods_alpha.scd1_agreements`
- `ods_alpha.scd1_agr_terms`
- `ods.scd1_z_r2_ip_merchants`
- `ods.scd1_z_r2_tariff_plan`
- `ods.scd1_z_r2_tariff_tune`
- `ods.scd1_z_r2_tariff_fix`
- `ods.scd1_z_r2_tariff_calc`
- `ods.scd1_z_r2_tariff_comiss` (join через `tt.c_tariff -> com.id`)
- `ods.scd1_z_r2_tariffs` (основной join `m.c_tariff_plan -> trf.c_tariff` + альтернативный `tt.c_tariff -> trf.id`)

In [ ]:
def run_debug_query(title, sql):
    print(f"\n===== {title} =====")
    try:
        with imp:
            imp.execute('set MEM_LIMIT=8g')
            df = imp.fetch(sql)
        if df is None or df.empty:
            print('No rows')
            return pd.DataFrame()
        print(f'Rows: {len(df):,}')
        display(df)
        return df
    except Exception as e:
        print(f'ERROR: {e}')
        return pd.DataFrame()


def run_full_debug_for_agr(debug_agr_id):
    sql_agreements_one = f"""
    select *
    from ods_alpha.scd1_agreements a
    where cast(a.abs_agr_id as string) = '{debug_agr_id}'
    """

    sql_agr_terms_one = f"""
    select *
    from ods_alpha.scd1_agr_terms t
    where cast(t.n_agr as string) in (
        select cast(a.n_agr as string)
        from ods_alpha.scd1_agreements a
        where cast(a.abs_agr_id as string) = '{debug_agr_id}'
    )
    """

    sql_r2_merchants_one = f"""
    select *
    from ods.scd1_z_r2_ip_merchants m
    where cast(m.id as string) = '{debug_agr_id}'
    """

    sql_r2_tariff_plan_one = f"""
    select tp.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_plan tp
      on cast(tp.id as string) = cast(m.c_tariff_plan as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    sql_r2_tariff_tune_one = f"""
    select tt.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt
      on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    sql_r2_tariff_fix_one = f"""
    select tf.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt
      on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
    left join ods.scd1_z_r2_tariff_fix tf
      on cast(tf.id as string) = cast(tt.c_tariff as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    sql_r2_tariff_calc_one = f"""
    select tc.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt
      on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
    left join ods.scd1_z_r2_tariff_calc tc
      on cast(tc.id as string) = cast(tt.c_tariff as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    sql_r2_tariff_comiss_one = f"""
    select
      cast(m.id as string) as agr_id,
      cast(m.c_tariff_plan as string) as c_tariff_plan,
      cast(tt.c_tariff as string) as tariff_id,
      com.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt
      on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
    left join ods.scd1_z_r2_tariff_comiss com
      on cast(com.id as string) = cast(tt.c_tariff as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    # Основной вариант для tariffs: tariff_plan -> trf.c_tariff
    sql_r2_tariffs_by_plan_one = f"""
    select
      cast(m.id as string) as agr_id,
      cast(m.c_tariff_plan as string) as c_tariff_plan,
      trf.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariffs trf
      on cast(trf.c_tariff as string) = cast(m.c_tariff_plan as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    # Альтернативный вариант для tariffs: tariff_id (tt.c_tariff) -> trf.id
    sql_r2_tariffs_by_id_one = f"""
    select
      cast(m.id as string) as agr_id,
      cast(m.c_tariff_plan as string) as c_tariff_plan,
      cast(tt.c_tariff as string) as tariff_id,
      trf.*
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt
      on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
    left join ods.scd1_z_r2_tariffs trf
      on cast(trf.id as string) = cast(tt.c_tariff as string)
    where cast(m.id as string) = '{debug_agr_id}'
    """

    print(f'\nDEBUG agr_id: {debug_agr_id}')
    run_debug_query('ods_alpha.scd1_agreements', sql_agreements_one)
    run_debug_query('ods_alpha.scd1_agr_terms', sql_agr_terms_one)
    run_debug_query('ods.scd1_z_r2_ip_merchants', sql_r2_merchants_one)
    run_debug_query('ods.scd1_z_r2_tariff_plan', sql_r2_tariff_plan_one)
    run_debug_query('ods.scd1_z_r2_tariff_tune', sql_r2_tariff_tune_one)
    run_debug_query('ods.scd1_z_r2_tariff_fix', sql_r2_tariff_fix_one)
    run_debug_query('ods.scd1_z_r2_tariff_calc', sql_r2_tariff_calc_one)
    run_debug_query('ods.scd1_z_r2_tariff_comiss (optional, by tt.c_tariff -> com.id)', sql_r2_tariff_comiss_one)
    run_debug_query('ods.scd1_z_r2_tariffs (optional, by m.c_tariff_plan -> trf.c_tariff)', sql_r2_tariffs_by_plan_one)
    run_debug_query('ods.scd1_z_r2_tariffs (optional alt, by tt.c_tariff -> trf.id)', sql_r2_tariffs_by_id_one)


for debug_agr_id in target_agr_ids:
    run_full_debug_for_agr(debug_agr_id)

print('\nГотово: сохранение в файлы отключено по запросу.')

mismatch_march_sample_10